# 03a — SHAP and LIME attributions for ML baselines

Generate and save SHAP and LIME explanations for the Random Forest and
LightGBM baselines on both the Elliptic Bitcoin and Ethereum Fraud
Detection datasets. Upstream models are loaded from notebook 02a/02b
without any retraining.

**Prerequisite:** notebooks 02a and 02b have been executed.

**Outputs saved under `experiments/results/`:**

- `shap_values_{rf,lgb}_{elliptic,ethereum}.npy` (4 files)
- `lime_attrs_{rf,lgb}_{elliptic,ethereum}.npy` (4 files)
- `xai_eval_indices_{elliptic,ethereum}.npy` (the balanced indices used
  for every downstream evaluation module)
- `shap_importance_{elliptic,ethereum}.csv`
- Figures: `shap_{elliptic,ethereum}_summary.png`,
  `local_explanations_elliptic.png`


In [ ]:
import os
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.xai.shap_wrapper import ShapTreeExplainer
from xai_blockchain_framework.xai.lime_wrapper import LimeTabularExplainer
from xai_blockchain_framework.utils.sampling import (
    sample_balanced,
    top_features,
    jaccard_topk,
)

set_seed(42)

RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'fraud': '#e74c3c', 'legit': '#27ae60', 'rf': '#2ecc71', 'lgb': '#9b59b6'}

# Sampling parameters shared with Modules 1-3
N_EXPLAIN = 200   # SHAP subset (also upper bound for LIME)
N_LIME = 100      # LIME is slower so we keep a smaller subset
SEED = 42

print("=" * 70)
print("03a  SHAP and LIME explainability (ML baselines)")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"N_EXPLAIN={N_EXPLAIN}, N_LIME={N_LIME}")


## 1. Load models and data splits


In [ ]:
print("\n" + "=" * 70)
print("1. Loading models and splits")
print("=" * 70)

# Elliptic
rf_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
ell_splits = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
X_train_ell = ell_splits['X_train']
X_test_ell = ell_splits['X_test']
y_test_ell = ell_splits['y_test']
fcols_ell = list(np.load(os.path.join(RESULTS_PATH, 'elliptic_feature_names.npy'), allow_pickle=True))
print(f"Elliptic  RF and LGB loaded; test={len(y_test_ell)}, features={len(fcols_ell)}")

# Ethereum
rf_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))
eth_splits = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_train_eth = eth_splits['X_train']
X_test_eth = eth_splits['X_test']
y_test_eth = eth_splits['y_test']
fcols_eth = list(np.load(os.path.join(RESULTS_PATH, 'ethereum_feature_names.npy'), allow_pickle=True))
print(f"Ethereum  RF and LGB loaded; test={len(y_test_eth)}, features={len(fcols_eth)}")


## 2. Balanced sampling

Select a class-balanced set of `N_EXPLAIN=200` indices per dataset (fraud
first, then legitimate). LIME reuses the first `N_LIME=100` of those
indices; the full set of 200 is persisted so that Modules 1, 2 and 3
all evaluate on identical instances.


In [ ]:
print("\n" + "=" * 70)
print("2. Sampling")
print("=" * 70)

eval_idx_ell = sample_balanced(y_test_ell, N_EXPLAIN, seed=SEED)
eval_idx_eth = sample_balanced(y_test_eth, N_EXPLAIN, seed=SEED)
lime_idx_ell = eval_idx_ell[:N_LIME]
lime_idx_eth = eval_idx_eth[:N_LIME]

# Persist indices for reuse in Modules 1-3
np.save(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'), eval_idx_ell)
np.save(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'), eval_idx_eth)

X_explain_ell = X_test_ell[eval_idx_ell]
y_explain_ell = y_test_ell[eval_idx_ell]
X_explain_eth = X_test_eth[eval_idx_eth]
y_explain_eth = y_test_eth[eval_idx_eth]

print(f"Elliptic SHAP subset: {len(eval_idx_ell)} ({int((y_explain_ell == 1).sum())} fraud)")
print(f"Elliptic LIME subset: {len(lime_idx_ell)}")
print(f"Ethereum SHAP subset: {len(eval_idx_eth)} ({int((y_explain_eth == 1).sum())} fraud)")
print(f"Ethereum LIME subset: {len(lime_idx_eth)}")


## 3. SHAP — tree explainer

For each tree-based classifier we build a `ShapTreeExplainer` (library
wrapper around `shap.TreeExplainer` with `check_additivity=False`) and
compute attributions on the SHAP subset. Values are saved as float32 for
downstream modules.


In [ ]:
print("\n" + "=" * 70)
print("3. SHAP TreeExplainer")
print("=" * 70)

def compute_shap(model, X_eval, name):
    explainer = ShapTreeExplainer(model)
    sv = explainer.explain(X_eval).astype(np.float32)
    print(f"  {name}: shape={sv.shape}")
    return sv, explainer

shap_rf_ell, exp_rf_ell = compute_shap(rf_ell, X_explain_ell, "RF-Elliptic")
shap_lgb_ell, _ = compute_shap(lgb_ell, X_explain_ell, "LGB-Elliptic")
shap_rf_eth, exp_rf_eth = compute_shap(rf_eth, X_explain_eth, "RF-Ethereum")
shap_lgb_eth, _ = compute_shap(lgb_eth, X_explain_eth, "LGB-Ethereum")

np.save(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'), shap_rf_ell)
np.save(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'), shap_lgb_ell)
np.save(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'), shap_rf_eth)
np.save(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'), shap_lgb_eth)
print("SHAP values saved.")


## 4. LIME — tabular explainer

LIME uses the library `LimeTabularExplainer`, which matches the research
code: `discretize_continuous=True` and greedy feature-name matching so
that attributions are densely aligned with the original feature order.


In [ ]:
print("\n" + "=" * 70)
print("4. LIME TabularExplainer")
print("=" * 70)

def compute_lime(model, X_train, X_eval, eval_indices, feature_names, name):
    explainer = LimeTabularExplainer(
        X_train, feature_names=feature_names, random_state=SEED
    )
    attrs = explainer.explain(
        X_eval,
        predict_proba=model.predict_proba,
        indices=eval_indices,
        verbose_every=25,
        name=name,
    ).astype(np.float32)
    print(f"  {name}: shape={attrs.shape}")
    return attrs

lime_rf_ell = compute_lime(rf_ell, X_train_ell, X_test_ell, lime_idx_ell, fcols_ell, "RF-Elliptic")
lime_lgb_ell = compute_lime(lgb_ell, X_train_ell, X_test_ell, lime_idx_ell, fcols_ell, "LGB-Elliptic")
lime_rf_eth = compute_lime(rf_eth, X_train_eth, X_test_eth, lime_idx_eth, fcols_eth, "RF-Ethereum")
lime_lgb_eth = compute_lime(lgb_eth, X_train_eth, X_test_eth, lime_idx_eth, fcols_eth, "LGB-Ethereum")

np.save(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'), lime_rf_ell)
np.save(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'), lime_lgb_ell)
np.save(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'), lime_rf_eth)
np.save(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'), lime_lgb_eth)
print("LIME attributions saved.")


## 5. SHAP summary plots

Standard SHAP summary (beeswarm) and bar plots for RF and LGB on both
datasets.


In [ ]:
print("\n" + "=" * 70)
print("5. SHAP summary plots")
print("=" * 70)

def _summary_grid(svs, X_explain, feature_names, title_prefix, path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    for ax, (sv, name) in zip(axes[0], svs):
        plt.sca(ax)
        shap.summary_plot(sv, X_explain, feature_names=feature_names,
                          show=False, max_display=15, plot_size=None)
        ax.set_title(f"{title_prefix}  {name} SHAP Summary")
    for ax, (sv, name) in zip(axes[1], svs):
        plt.sca(ax)
        shap.summary_plot(sv, X_explain, feature_names=feature_names,
                          plot_type='bar', show=False, max_display=15, plot_size=None)
        ax.set_title(f"{title_prefix}  {name} Feature Importance")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

_summary_grid(
    [(shap_rf_ell, 'RF'), (shap_lgb_ell, 'LGB')],
    X_explain_ell, fcols_ell, 'Elliptic',
    os.path.join(FIGURES_PATH, 'shap_elliptic_summary.png'),
)
_summary_grid(
    [(shap_rf_eth, 'RF'), (shap_lgb_eth, 'LGB')],
    X_explain_eth, fcols_eth, 'Ethereum',
    os.path.join(FIGURES_PATH, 'shap_ethereum_summary.png'),
)


## 6. Local explanations (Elliptic RF)

One fraud case and one legitimate case, side by side, explained by SHAP
(top) and LIME (bottom).


In [ ]:
print("\n" + "=" * 70)
print("6. Local explanations")
print("=" * 70)

fraud_idx = int(np.where(y_explain_ell == 1)[0][0])
legit_idx = int(np.where(y_explain_ell == 0)[0][0])
base_val = exp_rf_ell.expected_value

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plt.sca(axes[0, 0])
shap.waterfall_plot(shap.Explanation(
    values=shap_rf_ell[fraud_idx], base_values=base_val,
    data=X_explain_ell[fraud_idx], feature_names=fcols_ell,
), max_display=10, show=False)
axes[0, 0].set_title('Elliptic RF  FRAUD (SHAP)')

plt.sca(axes[0, 1])
shap.waterfall_plot(shap.Explanation(
    values=shap_rf_ell[legit_idx], base_values=base_val,
    data=X_explain_ell[legit_idx], feature_names=fcols_ell,
), max_display=10, show=False)
axes[0, 1].set_title('Elliptic RF  LEGITIMATE (SHAP)')

for ax_idx, (label, i) in enumerate([('FRAUD', 0), ('LEGITIMATE', N_LIME // 2)]):
    ax = axes[1, ax_idx]
    attrs = lime_rf_ell[i]
    top_idx = np.argsort(np.abs(attrs))[::-1][:10]
    fnames = [fcols_ell[int(j)] for j in top_idx]
    vals = attrs[top_idx]
    colors = [COLORS['fraud'] if v > 0 else COLORS['legit'] for v in vals]
    ax.barh(fnames, vals, color=colors)
    ax.set_xlabel('Contribution')
    ax.set_title(f'Elliptic RF  {label} (LIME)')
    ax.axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'local_explanations_elliptic.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 7. SHAP vs LIME top-feature overlap

Print the top-10 features from each method and compute the Jaccard
overlap, separately for the Elliptic and Ethereum RF models. Also
persist the per-feature SHAP importance as CSV.


In [ ]:
print("\n" + "=" * 70)
print("7. SHAP vs LIME (top features)")
print("=" * 70)

top_shap_ell = top_features(shap_rf_ell, fcols_ell, n=10)
top_lime_ell = top_features(lime_rf_ell, fcols_ell, n=10)
j_ell = jaccard_topk(top_shap_ell, top_lime_ell, k=10)

print("\n--- Elliptic RF: Top 10 ---")
print(f"{'Rank':<5} {'SHAP':<15} {'Imp':<10} {'LIME':<15} {'Imp':<10}")
print("-" * 55)
for i in range(10):
    sf, si = top_shap_ell[i]
    lf, li = top_lime_ell[i]
    print(f"{i+1:<5} {sf:<15} {si:<10.4f} {lf:<15} {li:<10.4f}")
print(f"Jaccard@10: {j_ell:.3f}")

top_shap_eth = top_features(shap_rf_eth, fcols_eth, n=10)
top_lime_eth = top_features(lime_rf_eth, fcols_eth, n=10)
j_eth = jaccard_topk(top_shap_eth, top_lime_eth, k=10)

print("\n--- Ethereum RF: Top 10 ---")
print(f"{'Rank':<5} {'SHAP':<35} {'Imp':<10} {'LIME':<35} {'Imp':<10}")
print("-" * 95)
for i in range(10):
    sf, si = top_shap_eth[i]
    lf, li = top_lime_eth[i]
    print(f"{i+1:<5} {sf:<35} {si:<10.4f} {lf:<35} {li:<10.4f}")
print(f"Jaccard@10: {j_eth:.3f}")

for name, sv, fnames in [
    ('elliptic', shap_rf_ell, fcols_ell),
    ('ethereum', shap_rf_eth, fcols_eth),
]:
    imp_df = pd.DataFrame({'feature': fnames, 'shap_rf': np.abs(sv).mean(0)})
    imp_df = imp_df.sort_values('shap_rf', ascending=False)
    imp_df.to_csv(os.path.join(RESULTS_PATH, f'shap_importance_{name}.csv'), index=False)


## 8. Summary


In [ ]:
print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print("""
Files saved:
  SHAP values  : shap_values_{rf,lgb}_{elliptic,ethereum}.npy (4 files)
  LIME attrs   : lime_attrs_{rf,lgb}_{elliptic,ethereum}.npy (4 files)
  Eval indices : xai_eval_indices_{elliptic,ethereum}.npy (2 files)
  Importance   : shap_importance_{elliptic,ethereum}.csv (2 files)
  Figures      : shap_*_summary.png, local_explanations_elliptic.png
""")
